<a href="https://colab.research.google.com/github/Mxnaal25/Starter-Notebooks/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mxnaal25/Starter-Notebooks/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis: One row = one unique content page for a specific client on a specific day.

Time Window: I am looking at March 2026 (month=2026-03). I am deliberately avoiding June 2026 because it is the final month and must be used as a sealed test set to avoid data leakage.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb
from google.colab import userdata

# Get the Hugging Face token securely from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Connect to DuckDB and set up the secret
con = duckdb.connect()
con.execute(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

# --- FIXED PATH WITH /**/*.parquet ---
query1 = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT(client_hash_id, '_', content_hash_id, '_', CAST(report_date AS VARCHAR))) AS unique_combinations
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03';
"""

print("--- Query 1: Verifying Grain ---")
display(con.execute(query1).df())

--- Query 1: Verifying Grain ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_combinations
0,9841378,9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature (Knowable at decision moment):

impressions: Observed traffic before the prediction date.

clicks: Observed clicks before the prediction date.

ctr: Calculated from clicks and impressions in the past.

position: Observed search ranking in the past.

word_count: Exists in the content metadata before prediction.

Label (What we want to predict):

is_declining: A binary proxy (1 or 0) representing if the page will lose more than 20% of its impressions in the next 30 days.

Context (Used for joining/filtering, not for the model):

client_hash_id, content_hash_id, report_date.

Excluded (Why):

health_score or needs_ctr_fix: These are pre-calculated by the system and directly contain the answer we want to predict (Data Leakage).

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# PROVE ROW COUNT AND DATE SPAN
query2 = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03';
"""

print("--- Query 2: Row Count and Date Span ---")
display(con.execute(query2).df())

--- Query 2: Row Count and Date Span ---


,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The third verification checks for availability. I need to ensure that the pages I'm looking at actually have traffic data, and aren't just empty rows.

I filter using impressions IS NOT NULL to see how many valid data points survive in my chosen month.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# PROVE AVAILABILITY (How many rows have actual impressions?)
query3 = """
SELECT
    COUNT(*) AS surviving_rows
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
  AND gsc_impressions IS NOT NULL;
"""

print("--- Query 3: Availability Check ---")
display(con.execute(query3).df())

--- Query 3: Availability Check ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,surviving_rows
0,9841378


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Limitation: This dataset is an unbalanced panel. Different clients started using the tool at different times (some have years of data, some have weeks). Because of this, calculating 30-day historical features might result in nulls or highly volatile numbers for newer clients. The model will need to handle these missing historical windows carefully.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell intentionally left blank. The limitation is documented in the text cell above.
print("Data limits documented.")


Data limits documented.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.